# `mdpp` Example: Conformational Clustering

This notebook demonstrates all clustering methods available in `mdpp`:

**Distance-matrix methods** (operate on a pairwise RMSD matrix):

- `Gromos` -- greedy largest-cluster-first (Numba JIT, O(n) aux memory)
- `Hierarchical` -- agglomerative clustering (scipy)
- `DBSCAN` -- density-based with noise detection (Numba JIT or sklearn)
- `HDBSCAN` -- hierarchical density-based (sklearn)

**Feature-vector methods** (operate on PCA/TICA projections):

- `KMeans` -- standard k-means (sklearn)
- `MiniBatchKMeans` -- scalable mini-batch variant (sklearn)
- `RegularSpace` -- regular-space discretization (deeptime)

Each method is a frozen dataclass configured at construction and called on data:

```python
result = Gromos(cutoff_nm=0.15)(rmsd_matrix)
result = KMeans(n_clusters=10)(pca.projections)
```

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from mplplots.utils import auto_ticks

from mdpp.analysis.clustering import (
    DBSCAN,
    HDBSCAN,
    Gromos,
    Hierarchical,
    KMeans,
    MiniBatchKMeans,
    RegularSpace,
    compute_rmsd_matrix,
)
from mdpp.analysis.decomposition import compute_pca, featurize_backbone_torsions
from mdpp.core.trajectory import align_trajectory, load_trajectory
from mdpp.plots import plot_cluster_populations, plot_feature_clustering

plt.style.use("mplplots.styles.GraphPadPrism")

In [ ]:
TOPOLOGY_PATH = Path("/path/to/topology.pdb")
TRAJECTORY_PATH = Path("/path/to/trajectory.xtc")
STRIDE = 5
CUTOFF_NM = 0.15

if not TOPOLOGY_PATH.exists() or not TRAJECTORY_PATH.exists():
    raise FileNotFoundError(
        "Update TOPOLOGY_PATH and TRAJECTORY_PATH before running analysis cells."
    )

traj = load_trajectory(
    trajectory_path=TRAJECTORY_PATH,
    topology_path=TOPOLOGY_PATH,
    stride=STRIDE,
    atom_selection="protein",
)
traj = align_trajectory(traj, atom_selection="name CA")

print(f"Frames: {traj.n_frames}, Atoms: {traj.n_atoms}")

## Compute RMSD Matrix

The pairwise RMSD matrix is shared by all distance-matrix clustering methods.
Use `backend="numba"` or `backend="torch"` for large trajectories.

In [ ]:
rmsd_mat = compute_rmsd_matrix(traj, atom_selection="backbone", backend="numba")

print(f"RMSD matrix: {rmsd_mat.rmsd_matrix_nm.shape}, dtype={rmsd_mat.rmsd_matrix_nm.dtype}")
print(f"Range: {rmsd_mat.rmsd_matrix_nm.max():.3f} nm")

## Distance-Matrix Methods

### GROMOS

Greedy largest-cluster-first assignment. Custom Numba kernel with O(n) auxiliary memory -- handles 120k+ frames.

In [ ]:
gromos = Gromos(cutoff_nm=CUTOFF_NM)(rmsd_mat.rmsd_matrix_nm)

print(f"GROMOS: {gromos.n_clusters} clusters")
for i in range(min(5, gromos.n_clusters)):
    count = int(np.sum(gromos.labels == i))
    print(f"  Cluster {i}: {count} frames, medoid={gromos.medoid_frames[i]}")

### Hierarchical

Agglomerative clustering via scipy. Supports `distance_threshold` (default) or fixed `n_clusters`.

In [ ]:
# Distance-threshold mode (like GROMOS cutoff)
hier_dist = Hierarchical(
    linkage_method="average",
    distance_threshold=CUTOFF_NM,
)(rmsd_mat.rmsd_matrix_nm)

# Fixed cluster count mode
hier_k = Hierarchical(
    linkage_method="average",
    n_clusters=5,
)(rmsd_mat.rmsd_matrix_nm)

print(f"Hierarchical (distance_threshold={CUTOFF_NM}): {hier_dist.n_clusters} clusters")
print(f"Hierarchical (n_clusters=5): {hier_k.n_clusters} clusters")

### DBSCAN

Density-based clustering with noise detection. Frames that don't belong to any dense region get label -1.

The default `backend="numba"` uses a custom Numba kernel with O(n) auxiliary memory. Pass `backend="sklearn"` for the official scikit-learn implementation.

In [ ]:
dbscan = DBSCAN(eps=CUTOFF_NM, min_samples=5)(rmsd_mat.rmsd_matrix_nm)

noise = int(np.sum(dbscan.labels == -1))
print(f"DBSCAN: {dbscan.n_clusters} clusters, {noise} noise frames")

# sklearn backend for comparison
dbscan_sk = DBSCAN(eps=CUTOFF_NM, min_samples=5, backend="sklearn")(rmsd_mat.rmsd_matrix_nm)
print(f"DBSCAN (sklearn): {dbscan_sk.n_clusters} clusters")

### HDBSCAN

Hierarchical density-based clustering via sklearn. Handles clusters of varying density without an epsilon parameter.

In [ ]:
hdbscan = HDBSCAN(min_cluster_size=10, min_samples=5)(rmsd_mat.rmsd_matrix_nm)

noise = int(np.sum(hdbscan.labels == -1))
print(f"HDBSCAN: {hdbscan.n_clusters} clusters, {noise} noise frames")

### Compare Distance-Matrix Methods

In [ ]:
results = {
    "GROMOS": gromos,
    "Hierarchical": hier_dist,
    "DBSCAN": dbscan,
    "HDBSCAN": hdbscan,
}

n_total = traj.n_frames
print(f"{'Method':<15s} {'Clusters':>10s} {'Noise':>8s} {'Largest':>10s}")
print("-" * 45)
for name, r in results.items():
    noise = int(np.sum(r.labels == -1))
    valid = r.labels[r.labels >= 0]
    largest = int(np.bincount(valid).max()) if len(valid) > 0 else 0
    pct = largest / n_total * 100
    print(f"{name:<15s} {r.n_clusters:>10d} {noise:>8d} {largest:>6d} ({pct:.1f}%)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=120, sharey=True)

for ax, (name, r) in zip(axes.ravel(), results.items()):
    plot_cluster_populations(r, top_k=20, ax=ax)
    ax.set_title(f"{name} ({r.n_clusters} clusters)")
    auto_ticks(ax)

axes[0, 0].set_ylabel("Frames")
axes[1, 0].set_ylabel("Frames")
fig.suptitle(f"Cluster Populations (cutoff = {CUTOFF_NM} nm)", y=1.02)
fig.tight_layout()

## Feature-Vector Methods

Backbone torsion featurization (sin/cos embedded phi/psi) followed by PCA.
Feature-based methods scale linearly with N and don't require the RMSD matrix.

In [ ]:
torsions = featurize_backbone_torsions(traj, atom_selection="protein")
pca = compute_pca(torsions.values, n_components=10)

print(f"Torsion features: {torsions.values.shape[1]}")
print(
    f"PCA: {pca.projections.shape[1]} components, "
    f"explained variance = {pca.explained_variance_ratio.sum():.1%}"
)

In [ ]:
N_CLUSTERS = 5

km = KMeans(n_clusters=N_CLUSTERS)(pca.projections)
mb = MiniBatchKMeans(n_clusters=N_CLUSTERS, batch_size=256)(pca.projections)
rs = RegularSpace(dmin=1.0)(pca.projections)

print(f"KMeans:         {km.n_clusters} clusters, inertia={km.inertia:.1f}")
print(f"MiniBatchKMeans: {mb.n_clusters} clusters, inertia={mb.inertia:.1f}")
print(f"RegularSpace:   {rs.n_clusters} clusters (dmin=1.0)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), dpi=120)

for ax, (name, r) in zip(axes, [("KMeans", km), ("MiniBatch", mb), ("RegularSpace", rs)]):
    plot_feature_clustering(r, pca, s=2, alpha=0.4, ax=ax)
    ax.set_title(f"{name} ({r.n_clusters} clusters)")

fig.tight_layout()

## Save Medoid Structures

Extract representative frames from the GROMOS result.

In [ ]:
output_dir = Path("cluster_medoids")
output_dir.mkdir(exist_ok=True)

for i, frame_idx in enumerate(gromos.medoid_frames[:10]):
    out = output_dir / f"cluster{i}_medoid.pdb"
    traj[int(frame_idx)].save(str(out))
    count = int(np.sum(gromos.labels == i))
    print(f"Cluster {i}: {count} frames, medoid frame {frame_idx} -> {out}")